# Semantic Entropy Probe (SEP)
Training notebook using the synthetic dataset.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)


## Load Dataset

In [ ]:
df = pd.read_csv("semantic_entropy_probe_synthetic_10000.csv")
df.head()


In [ ]:
FEATURES = [
    "hidden_mean",
    "hidden_std",
    "hidden_norm",
    "final_token_entropy",
    "max_logit",
    "logit_margin",
    "attention_dispersion",
    "retrieval_confidence",
    "semantic_consistency"
]

TARGET = "target_semantic_entropy"

X = df[FEATURES].values
y = df[TARGET].values.reshape(-1,1)

X_train,X_temp,y_train,y_temp = train_test_split(X,y,test_size=0.3,random_state=42)
X_val,X_test,y_val,y_test = train_test_split(X_temp,y_temp,test_size=0.5,random_state=42)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)


In [ ]:
class SEPDataset(Dataset):
    def __init__(self,X,y):
        self.X=torch.tensor(X,dtype=torch.float32)
        self.y=torch.tensor(y,dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self,idx):
        return self.X[idx],self.y[idx]

train_loader=DataLoader(SEPDataset(X_train,y_train),batch_size=64,shuffle=True)
val_loader=DataLoader(SEPDataset(X_val,y_val),batch_size=64)
test_loader=DataLoader(SEPDataset(X_test,y_test),batch_size=64)


## Model

In [ ]:
class SemanticEntropyProbe(nn.Module):
    def __init__(self,input_dim):
        super().__init__()
        self.net=nn.Sequential(
            nn.Linear(input_dim,64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64,32),
            nn.ReLU(),
            nn.Linear(32,1)
        )

    def forward(self,x):
        return self.net(x)

model=SemanticEntropyProbe(len(FEATURES)).to(device)

criterion=nn.MSELoss()
optimizer=torch.optim.AdamW(model.parameters(),lr=1e-3)
scheduler=torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer,patience=5)


## Train

In [ ]:
epochs=30
train_losses=[]
val_losses=[]
best=float("inf")

for epoch in range(epochs):
    model.train()
    tl=0
    for xb,yb in train_loader:
        xb,yb=xb.to(device),yb.to(device)
        optimizer.zero_grad()
        pred=model(xb)
        loss=criterion(pred,yb)
        loss.backward()
        optimizer.step()
        tl+=loss.item()

    tl/=len(train_loader)

    model.eval()
    vl=0
    with torch.no_grad():
        for xb,yb in val_loader:
            xb,yb=xb.to(device),yb.to(device)
            pred=model(xb)
            vl+=criterion(pred,yb).item()

    vl/=len(val_loader)

    scheduler.step(vl)

    train_losses.append(tl)
    val_losses.append(vl)

    if vl<best:
        best=vl
        torch.save(model.state_dict(),"sep_model.pt")

    print(epoch+1,tl,vl)


In [ ]:
plt.plot(train_losses,label="Train")
plt.plot(val_losses,label="Validation")
plt.legend()
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.show()


## Evaluation

In [ ]:
model.load_state_dict(torch.load("sep_model.pt",map_location=device))
model.eval()

preds=[]
truth=[]

with torch.no_grad():
    for xb,yb in test_loader:
        xb=xb.to(device)
        p=model(xb).cpu().numpy()
        preds.extend(p.flatten())
        truth.extend(yb.numpy().flatten())

print("MSE",mean_squared_error(truth,preds))
print("MAE",mean_absolute_error(truth,preds))
print("R2",r2_score(truth,preds))


## Inference

In [ ]:
sample=X_test[0]
sample_tensor=torch.tensor(sample,dtype=torch.float32).unsqueeze(0).to(device)

with torch.no_grad():
    score=model(sample_tensor).item()

print("Predicted Semantic Entropy:",score)
